# IESO Milestone 4
Serverless task entry point. Run through the versioned Databricks bundle.

In [ ]:
import json
import sys
from pathlib import Path

defaults = {
    "repo_root": "", "action": "probe", "mode": "kafka", "archive_record": "",
    "lmp_report_id": "", "lmp_expected_count": "", "totals_report_id": "", "totals_expected_count": "",
    "fault": "none", "source_revision": "", "job_run_id": "", "adf_run_id": "",
}
for name, default in defaults.items():
    dbutils.widgets.text(name, default)
params = {name: dbutils.widgets.get(name).strip() for name in defaults}

if not params["repo_root"].startswith("/"):
    raise ValueError("Run the deployed bundle job so repo_root resolves to an absolute workspace path.")
root = Path(params["repo_root"])
if not str(root).startswith("/Workspace/"):
    root = Path("/Workspace") / str(root).lstrip("/")
if not (root / "databricks/milestone4/lakehouse.py").is_file():
    raise ValueError(f"Missing uploaded source files under {root}")

sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "databricks"))

# Make the validation modules importable on Serverless workers.
pydeps = root / "databricks/milestone4/ieso_m4_pydeps.zip"

if not pydeps.is_file():
    raise FileNotFoundError(
        f"Missing worker Python dependency ZIP: {pydeps}"
    )

spark.addArtifacts(str(pydeps), pyfile=True)

print(f"Registered worker Python ZIP: {pydeps.name}")

from milestone4 import lakehouse as m4

spark.conf.set("spark.sql.session.timeZone", "UTC")
action, mode, fault = (params[name] for name in ("action", "mode", "fault"))
if mode not in ("kafka", "archive") or fault not in ("none", "silver_after_merge", "validation"):
    raise ValueError("Unknown mode or fault setting")
if action not in ("probe", "bronze", "silver", "validate"):
    raise ValueError("Unknown task action")

expected = {}
record = None
if mode == "archive":
    record = json.loads(params["archive_record"])
    source, report_id, _, _ = m4.obs.replay_input(record)
    expected[source] = (report_id, int(record["RowCount"]))
else:
    for prefix, source in (("lmp", "rt_lmp"), ("totals", "rt_totals")):
        report_id, count = params[f"{prefix}_report_id"], params[f"{prefix}_expected_count"]
        if report_id or count or action != "probe":
            m4.obs.require_hash(report_id)
            count = int(count)
            if not 0 < count <= 50_000:
                raise ValueError("Expected report row count must be between 1 and 50,000")
            expected[source] = (report_id, count)

print(json.dumps({"action": action, "mode": mode, "revision": params["source_revision"],
                  "job_run_id": params["job_run_id"], "adf_run_id": params["adf_run_id"]}))

if action == "probe":
    results = [m4.probe(spark, source, expected.get(source)) for source in m4.obs.TOPICS]
    print(json.dumps(results))
    if any(row.get("passed") is False for row in results):
        raise RuntimeError("Read-back counts do not match the selected reports")
elif action == "bronze":
    if mode == "archive":
        print(json.dumps(m4.land_archive(spark, record)))
    else:
        for source in expected:
            m4.ingest(spark, source)
elif action == "silver":
    for source in expected:
        print(json.dumps(m4.update_silver(spark, source, fault=fault)))
else:
    if fault == "validation":
        raise RuntimeError("IESO M4 intentional validation failure for the ADF integration test")
    m4.verify(spark, expected, mode, params["job_run_id"], params["adf_run_id"], params["source_revision"])
